In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from Custom_nn import NeuralNetwork
from PyTorch_nn import TorchModel


In [ ]:
data_path = os.path.join(os.getcwd(), 'data', 'Iris.csv')
df_iris = pd.read_csv(data_path)

df_iris.set_index('Id', inplace=True)

In [ ]:
display(df_iris)

### Brief Pre-processing

In [ ]:
X = df_iris.drop('Species', axis=1)
y = df_iris['Species']

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

std = StandardScaler()
X_train = std.fit_transform(X_train)
X_test  = std.transform(X_test)


### PyTorch Implementation

In [ ]:

# convert features to tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

#convert target to tensor
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)


In [ ]:
torch.manual_seed(42)
torch_model = TorchModel()


In [ ]:
# criterion setup
criterion = nn.CrossEntropyLoss()
#Choose optimizer --> Adam (popular), lr --> learning rate
optimizer = torch.optim.Adam(torch_model.parameters(), lr= 0.01)# the lower the lr, the longer it takes

#### Train Torch model

In [ ]:
# number of Epochs
epochs = 100
losses = [] # list of losses

for i in range(epochs):
    y_pred = torch_model.forward(X_train)
    #measure error
    loss = criterion(y_pred, y_train)
    losses.append(loss.detach().numpy())
    
    
    if i % 10 == 0:
        print(f'Epoch: {i} and loss: {loss}') # every 10 epochs
        
    #backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    

plt.plot(range(epochs), losses)
plt.ylabel("loss/epoch")
plt.xlabel('Epoch')
plt.show()

#### Evaluate Pytorch

In [ ]:
with torch.no_grad(): # turn off backprop
    y_eval = torch_model.forward(X_test)
    loss = criterion(y_eval, y_test) # find error in trained NN 
    
print(loss)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score

with torch.no_grad():
    y_logits = torch_model(X_test)
    y_pred = torch.argmax(y_logits, dim=1)

y_pred_np = y_pred.numpy()
y_test_np = y_test.numpy()

accuracy = accuracy_score(y_test_np, y_pred_np)
print(f"\nTest Accuracy: {accuracy:.4f}")

precision = precision_score(y_test_np, y_pred_np, average='macro')
print(f"Test Precision (macro-averaged): {precision:.4f}")